<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/Wafer_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
CLEAR SESSION

In [5]:
import tensorflow as tf
tf.keras.backend.clear_session()


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


MOUNT GOOGLE DRIVE

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Load Dataset Using TensorFlow

In [7]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"


In [8]:
#import os
#import shutil
#import random
#import math

# -------- CHANGE ONLY IF NEEDED --------
#TRAIN_DIR = "/content/drive/MyDrive/Datasets/train"
T#EST_DIR  = "/content/drive/MyDrive/Datasets/test"

# --------------------------------------

#os.makedirs(TEST_DIR, exist_ok=True)

#for class_name in os.listdir(TRAIN_DIR):

    #train_class_path = os.path.join(TRAIN_DIR, class_name)
    #test_class_path = os.path.join(TEST_DIR, class_name)

    #if not os.path.isdir(train_class_path):
        #continue

    #os.makedirs(test_class_path, exist_ok=True)

    #images = [img for img in os.listdir(train_class_path)
              #if img.lower().endswith((".jpg", ".jpeg", ".png"))]

    #total = len(images)
    #test_count = math.ceil(0.20 * total)   # 20% for test
    #train_count = total - test_count       # remaining 80%

    #print(f"\nClass: {class_name}")
    #print(f"Total images: {total}")
    #print(f"Train target: {train_count}")
    #print(f"Test target:  {test_count}")

    #selected_for_test = random.sample(images, test_count)

    #for img in selected_for_test:
        #src = os.path.join(train_class_path, img)
        #dst = os.path.join(test_class_path, img)
        #shutil.move(src, dst)

    #print(f"✅ Moved {test_count} images to test/{class_name}")

#print("\n🎯 80–20 split completed successfully!")


Class: bridge
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/bridge

Class: clean
Total images: 165
Train target: 132
Test target:  33
✅ Moved 33 images to test/clean

Class: cmp
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/cmp

Class: crack
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/crack

Class: ler
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/ler

Class: open
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/open

Class: vias
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/vias

Class: others
Total images: 162
Train target: 129
Test target:  33
✅ Moved 33 images to test/others

🎯 80–20 split completed successfully!


### Train the Model

IMPORT

In [8]:
import tensorflow as tf
import numpy as np
import os
from sklearn.utils import class_weight

DATA PATHS

In [9]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"


CREATE DATASETS (SAFE LOADING)

In [10]:
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)



Found 1032 files belonging to 8 classes.
Found 264 files belonging to 8 classes.


Get Class Names

In [11]:
class_names = train_data.class_names
print(class_names)


['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']


Normalize Images (Preprocessing)

In [12]:
def preprocess(image, label):
    image = image / 255.0                 # normalize
    image = tf.repeat(image, 3, axis=-1)  # gray → RGB
    return image, label

train_data = train_data.map(preprocess)
test_data  = test_data.map(preprocess)

In [13]:
for images, labels in train_data.take(1):
    print(images.shape)


(32, 224, 224, 3)


Load Pretrained Model (Transfer Learning)

In [15]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
base_model.trainable = False


Build Your Custom Model

In [17]:
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,016,312 (3.88 MB)

 Trainable params: 76,040 (297.03 KB)

 Non-trainable params: 940,272 (3.59 MB)

In [18]:
labels = np.concatenate([y for x,y in train_data], axis=0)

cw = class_weight.compute_class_weight(
    "balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(cw))
print("Class weights:", class_weights)


Class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0), 7: np.float64(1.0)}


FIRST EPOCH

In [19]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=12
)


Epoch 1/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 379ms/step - accuracy: 0.1737 - loss: 2.1907 - val_accuracy: 0.1250 - val_loss: 2.0812
Epoch 2/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 235ms/step - accuracy: 0.3503 - loss: 1.8074 - val_accuracy: 0.1402 - val_loss: 2.0563
Epoch 3/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 235ms/step - accuracy: 0.3893 - loss: 1.6865 - val_accuracy: 0.1894 - val_loss: 2.0309
Epoch 4/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - accuracy: 0.4705 - loss: 1.5823 - val_accuracy: 0.2727 - val_loss: 2.0037
Epoch 5/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - accuracy: 0.5209 - loss: 1.4797 - val_accuracy: 0.2879 - val_loss: 1.9727
Epoch 6/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - accuracy: 0.5437 - loss: 1.4108 - val_accuracy: 0.3030 - val_loss: 1.9374
Epoch 7/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 228ms/step - accuracy: 0.5236 - loss: 1.3586 - val_accuracy: 0.3371 - val_loss: 1.8970
Epoch 8/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 229ms/step - accuracy: 0.5785 - loss: 1.3066 - val_accuracy: 0

INCREASING TRAIN ACCURACY EPOCH 2

In [20]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-6),   # very gentle
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_more = model.fit(
    train_data,
    validation_data=test_data,
    epochs=6,
    callbacks=[early]
)


Epoch 1/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 261ms/step - accuracy: 0.6631 - loss: 1.0854 - val_accuracy: 0.6742 - val_loss: 1.5042
Epoch 2/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 229ms/step - accuracy: 0.6217 - loss: 1.1402 - val_accuracy: 0.7045 - val_loss: 1.4444
Epoch 3/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 229ms/step - accuracy: 0.6403 - loss: 1.1080 - val_accuracy: 0.7424 - val_loss: 1.3859
Epoch 4/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 227ms/step - accuracy: 0.6657 - loss: 1.0772 - val_accuracy: 0.7424 - val_loss: 1.3295
Epoch 5/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - accuracy: 0.6378 - loss: 1.0952 - val_accuracy: 0.7424 - val_loss: 1.2767
Epoch 6/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 229ms/step - accuracy: 0.6397 - loss: 1.1416 - val_accuracy: 0.7424 - val_loss: 1.2277


In [ ]:
EPOCH 3

In [21]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_more = model.fit(
    train_data,
    validation_data=test_data,
    epochs=4,
    callbacks=[early]
)


Epoch 1/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 268ms/step - accuracy: 0.6707 - loss: 1.1039 - val_accuracy: 0.7386 - val_loss: 1.1842
Epoch 2/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 233ms/step - accuracy: 0.6539 - loss: 1.0969 - val_accuracy: 0.7424 - val_loss: 1.1468
Epoch 3/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - accuracy: 0.6309 - loss: 1.1153 - val_accuracy: 0.7424 - val_loss: 1.1149
Epoch 4/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 244ms/step - accuracy: 0.6471 - loss: 1.0864 - val_accuracy: 0.7424 - val_loss: 1.0885


GETTING ALL LAYERS

In [22]:
for layer in model.layers[0].layers:
    if "conv" in layer.name.lower():
        print(layer.name)


conv
conv_bn
expanded_conv_depthwise_pad
expanded_conv_depthwise
expanded_conv_depthwise_bn
expanded_conv_squeeze_excite_avg_pool
expanded_conv_squeeze_excite_conv
expanded_conv_squeeze_excite_relu
expanded_conv_squeeze_excite_conv_1
expanded_conv_squeeze_excite_mul
expanded_conv_project
expanded_conv_project_bn
expanded_conv_1_expand
expanded_conv_1_expand_bn
expanded_conv_1_depthwise_pad
expanded_conv_1_depthwise
expanded_conv_1_depthwise_bn
expanded_conv_1_project
expanded_conv_1_project_bn
expanded_conv_2_expand
expanded_conv_2_expand_bn
expanded_conv_2_depthwise
expanded_conv_2_depthwise_bn
expanded_conv_2_project
expanded_conv_2_project_bn
expanded_conv_2_add
expanded_conv_3_expand
expanded_conv_3_expand_bn
expanded_conv_3_depthwise_pad
expanded_conv_3_depthwise
expanded_conv_3_depthwise_bn
expanded_conv_3_squeeze_excite_avg_pool
expanded_conv_3_squeeze_excite_conv
expanded_conv_3_squeeze_excite_relu
expanded_conv_3_squeeze_excite_conv_1
expanded_conv_3_squeeze_excite_mul
expande

LAST CONVOLUTION LAYER

In [26]:
last_conv = "conv_1"

SAVING THE MODEL

In [27]:
model.save("/content/drive/MyDrive/wafer_xai2_model.keras")
print("Saved XAI model")


Saved XAI model
